# RSGM samplers: Euler-Maruyama, the probability flow ODE (Euler, Heun) and the EDM schedule

The same RSGM 512x4 weights (their c10, seed 10), sampled in different ways. The question: how many network calls
(NFE) does RSGM need for the quality it has now with 100 steps of Euler-Maruyama, and what does the time per pose
become? A sampler that reaches it with fewer NFE moves RSGM to the left on the Pareto plots, with no retraining.

- **Samplers** (`RSGM.sampler`, see `hp/models/rsgm.py`):
  - `euler_maruyama`: the reverse SDE, first order, 1 NFE per step (the default, as the JAX code);
  - `euler`: the probability flow ODE dx/dt = -beta(t)/2 * score, first order, 1 NFE per step, no noise;
  - `heun`: the same ODE, second order, 2 NFE per step. On SO(3) the velocities are coordinates in the frame of x
    and are averaged as they are (Runge-Kutta-Munthe-Kaas of order 2 needs no commutator terms).
- **Schedules** (`RSGM.schedule`): the times of the steps from t = 1 down to eps.
  - `uniform` in t (the default);
  - `edm` (Karras et al. 2022): uniform in sigma^(1/rho) with sigma = sqrt(s(t)), the angular noise scale. It puts the
    steps where the noise is small and the score changes fast; rho = 7 is their choice for images.
- **The grid** is by NFE, so that Heun (2 calls per step) is compared at the same cost: steps = NFE / calls per step.
- **Quality**: Wasserstein (our harness, deg / joint), FID and APD (protocol of PoseD-Flow, 20 sets of 500), with the
  settings of `hp.evaluation.pareto`. The log-likelihood does not depend on the sampler and is not computed.
- **Cost**: the time of `model.sample(4096)` on this machine, as in the Pareto notebook.
- **Cache**: a folder of its own (`CACHE` below), so that these runs, evaluated without the likelihood, never mix with
  the ones of the Pareto notebook.

**What the oracle tests showed** (`tests/test_rsgm.py`, a model with the exact score of IGSO(3) data, s0 = 0.1): Heun
is second order (the error falls 4x per halving of the step), and with `edm` it draws the right distribution at
20 steps (40 NFE); the ODE with Euler needs ~1000 steps, Euler-Maruyama ~200 (its noise corrects errors); with
`edm` (rho = 7) the first-order samplers get worse, because the steps go to noise levels far below the scale of the
oracle's data. Real poses have finer structure than the oracle, so the answer here may differ.

In [ ]:
import sys
from pathlib import Path

# the code of the checkout this notebook is in (e.g. a worktree), before an editable install of another one
SRC = next(str(p / "src") for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "hp").is_dir())
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import matplotlib.pyplot as plt
import numpy as np
import torch

from hp.evaluation import pareto as P
from hp.evaluation.data import load_reference
from hp.models.rsgm import SAMPLERS

assert sys.modules["hp.evaluation.pareto"].__file__.startswith(SRC), \
    f"hp comes from {sys.modules['hp.evaluation.pareto'].__file__}: restart the kernel"

# ---- parameters ----------------------------------------------------------------------
RUN = "~/data/experiments/rsgm_posedflow_small_s10"  # RSGM 512x4, their c10, seed 10 (weights_ema)
NAME = "RSGM 512x4 c10"
SAMPLER_NAMES = ["euler_maruyama", "euler", "heun"]
SCHEDULES = {  # label: the overrides of RSGM
    "uniform": dict(schedule="uniform"),
    "edm rho=7": dict(schedule="edm", schedule_rho=7.0),
    # "edm rho=3": dict(schedule="edm", schedule_rho=3.0),
}
NFES = [10, 20, 50, 100, 200]  # network calls per sample; Heun takes NFE / 2 steps
DEFAULT = ("euler_maruyama", "uniform", 100)  # the sampler of the Pareto notebook

DEVICE = P.default_device()
BATCH = 4096  # the n of the timing
REPEATS = P.REPEATS
CACHE = P.CACHE.parent / "cache_samplers"
RECOMPUTE = set()  # names (see MODELS) to evaluate again; True: all
RECOMPUTE_RUNTIME = set()
# --------------------------------------------------------------------------------------


def model_name(sampler, schedule, nfe):
    return f"{NAME}: {sampler}, {schedule}, {nfe} NFE"


MODELS, CONFIG = {}, {}
for sampler in SAMPLER_NAMES:
    for schedule, overrides in SCHEDULES.items():
        for nfe in NFES:
            if nfe % SAMPLERS[sampler]:
                continue
            name = model_name(sampler, schedule, nfe)
            MODELS[name] = dict(short=name.split(": ")[1], run=RUN, sample_steps=nfe // SAMPLERS[sampler],
                                sampler=sampler, **overrides)
            CONFIG[name] = (sampler, schedule, nfe)
print(f"{len(MODELS)} samplers of one network, device {DEVICE}, cache {CACHE}")

## 1. Quality

The data as in the Pareto notebook. The models are built without `log_prob` (`has_log_prob = False`), so the harness
skips the likelihood: it does not depend on the sampler and is the slow part of the harness.

In [ ]:
zoo = P.Zoo(MODELS, DEVICE)
test = load_reference("test", P.ROTATIONS, "posedflow")
train = load_reference("train", P.ROTATIONS, "posedflow")
print(f"test {len(test):,}   train {len(train):,}")


def without_likelihood(model):
    model.has_log_prob = False
    return model


harness, paper = P.evaluate_quality(zoo, test, train, RECOMPUTE, build_wrapper=without_likelihood, cache=CACHE)

## 2. Cost: the time of `model.sample(4096)` on this machine

In [ ]:
runtime = P.measure_runtime(zoo, RECOMPUTE_RUNTIME, (BATCH,), REPEATS, cache=CACHE)

## 3. The table

Wasserstein ± the std over the harness repeats; FID and APD ± the half-width of the 95% CI over the 20 sets.

In [ ]:
METRICS = {  # name: (mean, error bar, unit, lower is better)
    "Wasserstein": (lambda l: harness[l]["wasserstein"]["generated"]["joint_mean_deg"]["mean"],
                    lambda l: harness[l]["wasserstein"]["generated"]["joint_mean_deg"]["std"], "deg / joint", True),
    "FID": (lambda l: paper[l]["FID"]["mean"], lambda l: paper[l]["FID"]["ci95"], "", True),
    "APD": (lambda l: paper[l]["APD"]["mean"], lambda l: paper[l]["APD"]["ci95"], "cm", False),
}
REAL = {
    "Wasserstein": next(iter(harness.values()))["wasserstein"]["contrast"]["joint_mean_deg"]["mean"],
    "FID": paper["real train poses"]["FID"]["mean"],
    "APD": paper["real train poses"]["APD"]["mean"],
}

rows = []
for name, labels in zoo.groups.items():
    label = labels[0]
    sampler, schedule, nfe = CONFIG[name]
    rows.append(dict(name=name, sampler=sampler, schedule=schedule, nfe=nfe, steps=MODELS[name]["sample_steps"],
                     us=runtime[name]["batches"][str(BATCH)]["us_per_pose"],
                     **{m: get(label) for m, (get, _, _, _) in METRICS.items()},
                     **{m + " err": err(label) for m, (_, err, _, _) in METRICS.items()}))

print(f"{'sampler':15s} {'schedule':10s} {'NFE':>4s} {'steps':>5s} {'us/pose':>8s} {'Wasserstein':>14s} {'FID':>16s} {'APD':>13s}")
for r in sorted(rows, key=lambda r: (r["sampler"], r["schedule"], r["nfe"])):
    mark = "  <- default" if (r["sampler"], r["schedule"], r["nfe"]) == DEFAULT else ""
    print(f"{r['sampler']:15s} {r['schedule']:10s} {r['nfe']:4d} {r['steps']:5d} {r['us']:8.1f} "
          f"{r['Wasserstein']:7.2f} ± {r['Wasserstein err']:4.2f} {r['FID']:.4f} ± {r['FID err']:.4f} "
          f"{r['APD']:6.2f} ± {r['APD err']:4.2f}{mark}")
print(f"{'real train poses':15s} {'':10s} {'':4s} {'':5s} {'':8s} {REAL['Wasserstein']:7.2f} {'':6s} {REAL['FID']:.4f} {'':8s} "
      f"{REAL['APD']:6.2f}")
print(f"us/pose: sample({BATCH}) on {P.chip_name(DEVICE)} ({DEVICE}), median of {REPEATS} runs after a warm-up")

## 4. Quality against NFE


One colour per sampler, one line style per schedule. The dotted line: real train poses; the star: the default
sampler (Euler-Maruyama, uniform, 100 steps). A sampler is as good as the default from the NFE where its line enters
the band of the default's error bar.

In [ ]:
COLORS = {"euler_maruyama": "#2a78d6", "euler": "#1baf7a", "heun": "#e34948"}  # fixed order, one hue per sampler
STYLES = ["-", "--", ":", "-."]  # the k-th schedule
LINESTYLE = {schedule: STYLES[k % len(STYLES)] for k, schedule in enumerate(SCHEDULES)}


def curves(rows):
    """{(sampler, schedule): rows sorted by NFE}."""
    out = {}
    for r in sorted(rows, key=lambda r: r["nfe"]):
        out.setdefault((r["sampler"], r["schedule"]), []).append(r)
    return out


default = next(r for r in rows if (r["sampler"], r["schedule"], r["nfe"]) == DEFAULT)
with plt.style.context("default"):
    fig, axes = plt.subplots(1, len(METRICS), figsize=(6 * len(METRICS), 4.8))
    for ax, (metric, (_, _, unit, lower_better)) in zip(axes, METRICS.items()):
        ax.axhspan(default[metric] - default[metric + " err"], default[metric] + default[metric + " err"],
                   color="0.9", zorder=0)
        ax.axhline(REAL[metric], color="0.5", lw=1, ls=":", zorder=1)
        for (sampler, schedule), rs in curves(rows).items():
            ax.errorbar([r["nfe"] for r in rs], [r[metric] for r in rs], yerr=[r[metric + " err"] for r in rs],
                        color=COLORS[sampler], ls=LINESTYLE[schedule], lw=1.6, marker="o", ms=5, capsize=2,
                        label=f"{sampler}, {schedule}")
        ax.plot(default["nfe"], default[metric], marker="*", ms=14, color="black", ls="", zorder=5, label="default")
        ax.set_xscale("log")
        ax.set_xticks(NFES, [str(n) for n in NFES])
        ax.set_xlabel("network calls per sample (NFE)")
        ax.set_ylabel(f"{metric} ({unit})" if unit else metric)
        ax.set_title(f"{metric}, {'lower' if lower_better else 'higher (diversity; real poses dotted)'} is better",
                     fontsize=10)
        ax.grid(alpha=0.3)
    axes[0].legend(fontsize=8, loc="upper right")
    fig.suptitle(f"{NAME}: samplers of the same weights", fontsize=12)
    fig.tight_layout()
    plt.show()

## 5. Quality against time per pose (as the Pareto plots)

The quality on x (the better side on the left), the time per pose on y (log scale); each line runs over the NFE of
one sampler and schedule, and the dashed grey line is the Pareto front over all of them.

In [ ]:
def pareto(rows, metric, lower_better):
    sign = 1 if lower_better else -1
    front, best = [], np.inf
    for r in sorted(rows, key=lambda r: (r["us"], sign * r[metric])):
        if sign * r[metric] < best:
            front.append(r)
            best = sign * r[metric]
    return front


with plt.style.context("default"):
    fig, axes = plt.subplots(1, len(METRICS), figsize=(6 * len(METRICS), 5), sharey=True)
    for ax, (metric, (_, _, unit, lower_better)) in zip(axes, METRICS.items()):
        ax.axvline(REAL[metric], color="0.5", lw=1, ls=":", zorder=0)
        front = pareto(rows, metric, lower_better)
        ax.plot([r[metric] for r in front], [r["us"] for r in front], ls="--", lw=1.2, color="0.4", zorder=1)
        for (sampler, schedule), rs in curves(rows).items():
            ax.plot([r[metric] for r in rs], [r["us"] for r in rs], color=COLORS[sampler], ls=LINESTYLE[schedule],
                    lw=1.4, marker="o", ms=6, label=f"{sampler}, {schedule}", zorder=2)
            for r in rs:
                ax.annotate(str(r["nfe"]), (r[metric], r["us"]), xytext=(4, 3), textcoords="offset points",
                            fontsize=7, color="0.3")
        ax.plot(default[metric], default["us"], marker="*", ms=14, color="black", ls="", zorder=5, label="default")
        ax.set_yscale("log")
        if not lower_better:
            ax.invert_xaxis()
        ax.set_xlabel(f"{metric} ({unit})" if unit else metric)
        ax.grid(alpha=0.3, which="both")
    axes[0].set_ylabel(f"us per pose, sample({BATCH})")
    axes[0].legend(fontsize=8)
    fig.suptitle(f"{NAME}: time against quality; the numbers are the NFE", fontsize=12)
    fig.tight_layout()
    plt.show()

## 6. Reading

To fill in after the run. The questions:
- From which NFE is each sampler within the error bar of the default (100 NFE of Euler-Maruyama)? That NFE over 100 is
  the speed-up RSGM gets for free on the Pareto plots.
- Does the ODE (no noise) lose diversity (APD) or Wasserstein against the SDE at the same NFE?
- Does `edm` help the second-order sampler and hurt the first-order ones, as on the oracle? If `edm` with rho = 7 is
  bad, try rho = 3 (in `SCHEDULES`).
- Is the time linear in NFE (Heun's extra `exp_frame` per step is cheap against the network)?